In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Carichiamo il dataset
df = pd.read_csv("../data/raw/demand.csv")

# Order_Demand in numero: "(5)" diventa -5
df["Order_Demand"] = (
    df["Order_Demand"]
    .astype(str)
    .str.strip()
    .str.replace("(", "-", regex=False)
    .str.replace(")", "", regex=False)
)
df["Order_Demand"] = pd.to_numeric(df["Order_Demand"])

# Date in data vera, e teniamo solo il periodo affidabile
df["Date"] = pd.to_datetime(df["Date"], format="%Y/%m/%d")
df_analisi = df[(df["Date"] >= "2012-01-01") & (df["Date"] <= "2016-12-31")].copy()

In [4]:
# Somma della domanda per mese 
df_analisi["Mese"] = df_analisi["Date"].dt.to_period("M")
serie = df_analisi.groupby("Mese")["Order_Demand"].sum()

print("Mesi:", len(serie))

Mesi: 60


## Divisione tra training e test
Il training (2012-2015, 48 mesi) è il passato che conosciamo.
Il test (2016, 12 mesi) lo usiamo solo per controllare se le previsioni sono giuste.

In [6]:
# Training: tutti i mesi prima del 2016. Test: i 12 mesi del 2016
train = serie[serie.index < "2016-01"]
test = serie[serie.index >= "2016-01"]

print("Training:", len(train), "mesi")
print("Test:    ", len(test), "mesi")

Training: 48 mesi
Test:     12 mesi


## Metodo semplice: media degli ultimi 3 mesi
La previsione di ogni mese è la media dei 3 mesi precedenti. Non impara nulla dai dati:
ci serve come riferimento da battere.

In [7]:
media_mobile = serie.rolling(3).mean().shift(1)

previsione_base = media_mobile.loc[test.index]

In [8]:
# Errore percentuale di ogni mese: |previsto - reale| / reale * 100
errore = (previsione_base - test).abs() / test.abs() * 100

# Tabella di confronto: reale, previsto e errore di ogni mese del 2016
confronto = pd.DataFrame({
    "Reale": test,
    "Previsto": previsione_base.round(0),
    "Errore %": errore.round(1),
})
print(confronto)

# Errore medio sui 12 mesi: si chiama MAPE
mape_base = errore.mean()
print("MAPE della media mobile:", round(mape_base, 1), "%")

            Reale    Previsto  Errore %
Mese                                   
2016-01  78489871  85731825.0       9.2
2016-02  73794187  82431479.0      11.7
2016-03  93068160  80250728.0      13.8
2016-04  78731714  81784073.0       3.9
2016-05  79244535  81864687.0       3.3
2016-06  81996423  83681470.0       2.1
2016-07  88373342  79990891.0       9.5
2016-08  80049428  83204767.0       3.9
2016-09  77610066  83473064.0       7.6
2016-10  83854187  82010945.0       2.2
2016-11  89849906  80504560.0      10.4
2016-12  80306310  83771386.0       4.3
MAPE della media mobile: 6.8 %


### Risultato del metodo semplice

La media mobile a 3 mesi sbaglia in media del **6,8%** (MAPE) sui 12 mesi del 2016.

- **Mesi migliori:** giugno (2,1%) e ottobre (2,2%).
- **Mesi peggiori:** marzo (13,8%), febbraio (11,7%) e novembre (10,4%).
- La media mobile reagisce in ritardo ai cambi di domanda: a marzo 2016 la domanda è salita
  a 93 milioni di pezzi, ma la previsione, basata sui 3 mesi precedenti (più bassi), era di 80 milioni.
